# 01 · Data audit, evaluation harness and pipeline smoke test

**CS6366 final project:** object detection on VisDrone drone imagery.

This notebook is the project's first stage. It:

1. downloads VisDrone2019-DET through Ultralytics;
2. audits the labels: classes, objects per image, object sizes and ignored regions;
3. fine-tunes a pretrained YOLO11 model on a small share of one epoch, to check the pipeline and time an epoch on the current GPU;
4. evaluates that model two ways: with Ultralytics' built-in mAP, and with our VisDrone-aware COCO evaluation (`src/visdrone_eval.py`).

The smoke-test model sees only a share of one epoch, so its accuracy is **not a result**. It only shows that every step runs.

Run it on Colab with a GPU runtime (*Runtime → Change runtime type → GPU*). The first run downloads about 2 GB.

In [ ]:
# Setup: on Colab, fetch this repository and install its pinned requirements
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/gizaquoc197/cs6366-drone-detection"
if "google.colab" in sys.modules:
    if not Path("cs6366-drone-detection").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    REPO = Path("cs6366-drone-detection").resolve()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
else:  # running locally from a clone
    REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
print("Repository:", REPO)

In [ ]:
import json, time, urllib.request, zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from PIL import Image
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset, img2label_paths
from ultralytics.utils import ASSETS_URL

import visdrone_eval as ve

settings.update({"sync": False})  # turn off Ultralytics' anonymous usage analytics

DATA_YAML = os.environ.get("VISDRONE_YAML", "VisDrone.yaml")  # override only for local tests
MODEL = "yolo11n.pt"   # YOLO11 nano, pretrained on COCO
IMGSZ = 640            # Ultralytics' default input size
SEED = 0
SMOKE_FRACTION = float(os.environ.get("SMOKE_FRACTION", 0.1))  # share of training images in the smoke test
RUNS = REPO / "runs"

# Chart style: one blue for single series, orange as the second series, recessive axes
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, MUTED, SURFACE, GRID = "#0b0b0b", "#52514e", "#898781", "#fcfcfb", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK_2, "axes.titlecolor": INK, "axes.titlelocation": "left",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 10,
})

In [ ]:
# Download VisDrone and convert it to YOLO format (first run only)
data = check_det_dataset(DATA_YAML)
NAMES = data["names"]
SPLITS = [s for s in ("train", "val", "test") if data.get(s)]
print("Dataset root:", data["path"])
print("Splits:", SPLITS, "| classes:", list(NAMES.values()))

In [ ]:
# Our evaluator needs the raw VisDrone annotations, which the Ultralytics conversion deletes
RAW_ROOT = Path(data["path"]).parent / "VisDrone-raw"
RAW_VAL = RAW_ROOT / "VisDrone2019-DET-val"
if not RAW_VAL.exists():
    RAW_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = RAW_ROOT / "VisDrone2019-DET-val.zip"
    urllib.request.urlretrieve(f"{ASSETS_URL}/VisDrone2019-DET-val.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW_ROOT)
    zip_path.unlink()
gt_val = ve.load_visdrone_gt(RAW_VAL)
print(len(gt_val["images"]), "validation images with raw annotations")

## 1. Data audit

We read every label file, measure each object in original-image pixels, and use the COCO size bins: *small* objects are under 32×32 px in area, *medium* under 96×96 px, and *large* the rest.

YOLO shrinks each image so its long side equals the input size. The last columns show how many objects end up smaller than 8 px, the stride of YOLO11's finest detection layer, at input sizes 640 and 1280.

In [ ]:
img_rows, obj_rows = [], []
for split in SPLITS:
    dirs = data[split] if isinstance(data[split], list) else [data[split]]
    imgs = sorted(p for d in dirs for p in Path(d).glob("*.jpg"))
    for img_path, lbl_path in zip(imgs, img2label_paths([str(p) for p in imgs])):
        W, H = Image.open(img_path).size
        lbl_path = Path(lbl_path)
        lines = [l.split() for l in lbl_path.read_text().splitlines() if l.strip()] if lbl_path.exists() else []
        img_rows.append({"split": split, "image": img_path.stem, "objects": len(lines)})
        obj_rows += [{"split": split, "cls": int(c), "w": float(w) * W, "h": float(h) * H, "long_side": max(W, H)}
                     for c, _, _, w, h in lines]

images, objects = pd.DataFrame(img_rows), pd.DataFrame(obj_rows)
objects["area"] = objects.w * objects.h
objects["size_px"] = np.sqrt(objects.area)  # side of a square with the same area
objects["size_bin"] = pd.cut(objects.area, [0, 32**2, 96**2, np.inf], right=False, labels=["small", "medium", "large"])
for s in (640, 1280):
    objects[f"size_at_{s}"] = objects.size_px * s / objects.long_side

summary = images.groupby("split", sort=False).agg(
    images=("image", "size"), objects=("objects", "sum"),
    mean_per_image=("objects", "mean"), max_per_image=("objects", "max"),
    images_without_objects=("objects", lambda n: int((n == 0).sum())))
by_split = objects.groupby("split", sort=False)
for b in ("small", "medium", "large"):
    summary[f"{b} %"] = by_split.size_bin.apply(lambda x: 100 * (x == b).mean())
for s in (640, 1280):
    summary[f"< 8 px at {s} %"] = by_split[f"size_at_{s}"].apply(lambda x: 100 * (x < 8).mean())
summary.round(1)

In [ ]:
class_counts = objects.groupby(["cls", "split"]).size().unstack(fill_value=0)[SPLITS]
class_counts.index = [NAMES[i] for i in class_counts.index]
class_counts["share of all %"] = (100 * class_counts.sum(axis=1) / class_counts.values.sum()).round(1)
class_counts

In [ ]:
# Check that the raw annotations and the YOLO labels describe the same validation objects
n_raw = sum(1 for a in gt_val["annotations"] if not a["iscrowd"])
n_yolo = int(images.query("split == 'val'").objects.sum())
n_ignored = sum(1 for a in gt_val["annotations"] if a["iscrowd"] and a["category_id"] == 1)
print(f"Validation objects: raw {n_raw:,} vs YOLO labels {n_yolo:,} -> {'match' if n_raw == n_yolo else 'MISMATCH'}")
print(f"Validation ignored regions dropped by the Ultralytics conversion: {n_ignored:,}")

In [ ]:
train = objects[objects.split == SPLITS[0]]

fig, ax = plt.subplots(figsize=(7, 3.8))
counts = train.cls.map(NAMES).value_counts().sort_values()
ax.barh(counts.index, counts.values, color=BLUE, height=0.6)
ax.set_title(f"Labelled objects per class, {SPLITS[0]} split")
ax.set_xlabel("Objects")
ax.grid(axis="y", visible=False)
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
bins = np.logspace(0, np.log10(train.size_px.max() + 1), 50)
ax.hist(train.size_px, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=0.8)
ax.set_xscale("log")
for t in (32, 96):
    ax.axvline(t, color=MUTED, ls="--", lw=1)
top = ax.get_ylim()[1]
for x, label in ((np.sqrt(32 * 4), "small"), (np.sqrt(32 * 96), "medium"), (96 * 3, "large")):
    ax.text(x, top * 0.95, label, ha="center", va="top", color=INK_2)
ax.set_title(f"Object size in original pixels, {SPLITS[0]} split")
ax.set_xlabel("Object size (square root of box area, px, log scale)")
ax.set_ylabel("Objects")
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
xs = np.logspace(0, 2.5, 200)
for s, color in ((640, BLUE), (1280, ORANGE)):
    v = np.sort(train[f"size_at_{s}"].values)
    share = 100 * np.searchsorted(v, xs) / len(v)
    ax.plot(xs, share, color=color, lw=2, label=f"input size {s}")
    i = np.searchsorted(share, 70)  # label each curve where the two are well apart
    ax.text(xs[i] / 1.08, 70, f"{s}", color=INK_2, ha="right", va="center")
ax.axvline(8, color=MUTED, ls="--", lw=1)
ax.text(7.5, 60, "8 px: finest\nYOLO11 stride", color=INK_2, ha="right", va="center")
ax.set_xscale("log")
ax.set_ylim(0, 102)
ax.set_title("Share of objects smaller than a given size after resizing")
ax.set_xlabel("Object size after resizing (px, log scale)")
ax.set_ylabel("Objects at or below this size (%)")
ax.legend(frameon=False, loc="upper left")
plt.show()

In [ ]:
# Two validation images with their labels: the most crowded one and the one with most ignored regions
anns_by_img = {}
for a in gt_val["annotations"]:
    if not a["iscrowd"] or a["category_id"] == 1:  # draw each ignored region once
        anns_by_img.setdefault(a["image_id"], []).append(a)
n_obj = {i: sum(not a["iscrowd"] for a in v) for i, v in anns_by_img.items()}
n_ign = {i: sum(a["iscrowd"] for a in v) for i, v in anns_by_img.items()}
picks = [max(n_obj, key=n_obj.get), max(n_ign, key=n_ign.get)]
info = {im["id"]: im for im in gt_val["images"]}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), layout="constrained")
for ax, img_id in zip(axes, picks):
    ax.imshow(Image.open(RAW_VAL / "images" / info[img_id]["file_name"]))
    for a in anns_by_img[img_id]:
        x, y, w, h = a["bbox"]
        if a["iscrowd"]:
            ax.add_patch(Rectangle((x, y), w, h, facecolor=(1, 1, 1, 0.35), edgecolor="white", hatch="//", lw=0.8))
        else:
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=ORANGE, lw=0.8))
    ax.set_title(f"{info[img_id]['file_name']}: {n_obj[img_id]} objects, {n_ign[img_id]} ignored regions", fontsize=9)
    ax.axis("off")
fig.legend(handles=[Patch(fill=False, edgecolor=ORANGE, label="labelled object"),
                    Patch(facecolor=(1, 1, 1, 0.35), edgecolor="grey", hatch="//", label="ignored region")],
           loc="outside lower center", ncol=2, frameon=False)
plt.show()

## 2. Smoke test: fine-tune briefly and time it

This trains on a share of one epoch to check the pipeline and to estimate the cost of a full epoch on this GPU, before we plan the real experiments.

In [ ]:
model = YOLO(MODEL)
t0 = time.time()
model.train(data=DATA_YAML, epochs=1, imgsz=IMGSZ, fraction=SMOKE_FRACTION, seed=SEED, deterministic=True,
            project=str(RUNS), name="smoke", exist_ok=True, plots=False)
elapsed = time.time() - t0
SMOKE_WEIGHTS = Path(model.trainer.last)
print(f"{elapsed / 60:.1f} min for {SMOKE_FRACTION:.0%} of one epoch, including set-up and a validation pass")
print(f"Rough upper bound for one full epoch on this hardware: {elapsed / SMOKE_FRACTION / 60:.0f} min")

## 3. Evaluation harness

We evaluate on the validation split, with the same model and the same predictions, in two ways:

* **Ultralytics built-in mAP.** Its labels have no ignored regions, so a detection inside an unlabelled crowd counts as a false positive.
* **VisDrone-aware COCO evaluation** (`src/visdrone_eval.py`). It rebuilds ground truth from the raw annotations, treats ignored regions as COCO crowd regions so that detections there are neither rewarded nor penalised, allows up to 500 detections per image as VisDrone does, and reports AP by object size and by class.

The test-dev split is held out: we evaluate on it once, at the end, for the final numbers.

In [ ]:
smoke = YOLO(SMOKE_WEIGHTS)
val = smoke.val(data=DATA_YAML, split="val", imgsz=IMGSZ, batch=16, conf=0.001, iou=0.7, max_det=500,
                save_json=True, plots=False, project=str(RUNS), name="smoke_val", exist_ok=True)
dets = ve.load_ultralytics_predictions(Path(val.save_dir) / "predictions.json", gt_val)
ours = ve.evaluate(gt_val, dets)

pd.DataFrame({
    "Ultralytics built-in": {"mAP50": val.box.map50, "mAP50-95": val.box.map},
    "VisDrone-aware COCO": {"mAP50": ours["AP50"], "mAP50-95": ours["AP50-95"]},
}).round(4)

In [ ]:
pd.DataFrame({
    "AP50-95": {"small": ours["AP_small"], "medium": ours["AP_medium"], "large": ours["AP_large"]},
}).round(4)

In [ ]:
pd.Series(ours["per_class_AP50-95"], name="AP50-95").round(4).to_frame()

## 4. Next steps

* **Baseline.** Fine-tune YOLO11n on the full training split with a fixed schedule and seed, and evaluate it with the harness above.
* **One controlled experiment.** Change one design choice and keep everything else fixed. Input size, 640 against 1280, is the leading candidate, given how many objects fall below 8 px at 640.
* **Uncertainty.** Bootstrap confidence intervals over validation images, and repeat runs with 2–3 seeds if GPU time allows, so that we only report differences larger than the noise.
* **Error analysis.** Per-class and per-size AP, a confusion matrix, and examples of missed and false detections.
* **Final numbers.** Evaluate on the held-out test-dev split once, after all choices are fixed.